DELTA LAKE 
- Gives you transactional guarantees, Schema Enforcement, versinooing and time travel , your pipeline are reproducable and save




UNITY CATALOG 
- How we organizes data into Catalogs, Schemas and tables while handling Governance , security and lineage across your assets

Declarative Pipelines and Medallions Architecture
- Bronze, Silver, Gold Pattern, Powerful way to Structural transformations 
- Build Each Layer Step By Step Ingesting raw data Cleaning and standardized it and finally creating business ready aggregations

- After Building the Declarative Pipeline we'll Productinoize it by Scheduling and Orchestrating using Datarbicks Jobs
- Trun the Pipleine into a Realiable Production Workflow

LakeFlow Designer in SQL Editor 
- Extend Pipeline Visually 
- No Code Visual interface 


# DataSet exploration

In [0]:
%sql
select * from lakehouse_demo.default.online_retail

In [0]:
df = spark.table("lakehouse_demo.default.online_retail")
df.show(5)

In [0]:
df.limit(5).display()

In [0]:
df.count()

In [0]:
df.distinct().count()

In [0]:
df.printSchema()

In [0]:
display(df.dtypes)

In [0]:
df.describe().display()

In [0]:
%sql
select 
    substring(InvoiceDate, 1, 7) as month,
    sum(Quantity * UnitPrice) as revenue
from lakehouse_demo.default.online_retail
where Country = 'France'
group by month
order by month

Databricks visualization. Run in Databricks to view.

Transactional Data continouous flowing into your data data lake from a hundred Stores , 24 hours a day
- storing data in a data lake sound simple , But as soon as you Operate at scale problems start appearing

Tradational data lake 
- Failing Job leads to partial files
- two process writing files at the same time 
- Accidental file deletes 
- Shcema changes are not enforced 
- old data gets overwritten 

Only Store files
- Don't provide (Transactions, Consistency guarantees, versionoing)

That problem solve by Delta Lake 
- Single source of truth
- Every change is recorded
- Automatic Rollbacks
- no File Corruption
- Shcema changes are enforced and tracked 
- Query Specific Version

Delta Lake Convert data lake , it behaves like a realiable database

- Delta Lake Storage layer build on top of Parquet files, it uses parquet as the underlying storage format , Every Single write you perform whether it insert, Update, or delete or Merge Ultimately reuslts in a new parquet file being written a storage

- Physically data lives in parquet files
- Delta Lake adds top of that is transaction logs
- Tracks which files belong to a table version
- Logs which files are no longer active
- files are never modified 
- Copy on write approach 

Delta lake never modifies parquet files in a place. it uses a copy on write approach. Whenever data changes new parquet files are written and transaction log is updated to reflect the new state of table.

- That design enables ACID transactions, scalable metadata handling, and Unified batch and streaming processing, all while keeping the underlying storage simple and efficient 

- Every table in databricks is Delta table by default (Parquet files, _Delta_logs_Folder)
- JSON files that track every change made to the table

Each row represents one committed transaction, which corresponds to a JSON file in the transaction log

In Production Delta tables often stored in external cloud Storage (s3, ADLS, GCS)

- Delta Lake enforce the Schema on write , Every write must match the table defined schema , write fails if unexpected columns appears, if data type do not match or if required Columns are missings 
this prevents silent schema drift and protects downstream tables from inconsistent data 

- if Bronze data does not match the expected silver schema , fails immedaitely rather than silently corrupting the dataset 

TIME TRAVEL VERSIONING 

- Delta lake uses copy on write and nver modifies parquet , every commited creates a new version that means you can query the table as it existed at specific time version no or timestamp. this is exteremly useful for compliance audits, debugging erros , restroing accidently delted data , or analyzing historcal states before major business changes.

Since old Parquet files remain on disk, storage useage grows over time, right? 
Vaccum command can removed outdated files after a defined retentation period.





Catalog
- manged tables, unmanged tables (S3, GCS, ADLS (folders, tables (bronze, silver, Gold) Volumes )) Daily, monthly, weekly partiton

In [0]:
%sql
DESCRIBE HISTORY lakehouse_demo.default.online_retail

In [0]:
%sql
Use catalog lakehouse_demo;

In [0]:
%sql 
CREATE TABLE employees
  (id INT, name STRING, salary DOUBLE);

In [0]:
%sql
INSERT INTO employees
VALUES 
  (1, "Adam", 3500.0),
  (2, "Sarah", 4020.5);

INSERT INTO employees
VALUES
  (3, "John", 2999.3),
  (4, "Thomas", 4000.3);

INSERT INTO employees
VALUES
  (5, "Anna", 2500.0);

INSERT INTO employees
VALUES
  (6, "Kim", 6200.3)

In [0]:
%sql
select * from employees

In [0]:
%sql
DESCRIBE DETAIL employees

In [0]:
%sql
DESCRIBE DETAIL lakehouse_demo.default.employees


In [0]:
%sql 
DESCRIBE HISTORY lakehouse_demo.default.employees

In [0]:
# Pass the full catalog table path directly into the history method
df_history = spark.sql("DESCRIBE HISTORY lakehouse_demo.default.employees")

# Display it in a clean, interactive grid format
display(df_history)




In [0]:
%sql
-- Read the log state at a specific transaction version
SELECT * FROM lakehouse_demo.default.employees VERSION AS OF 5;


In [0]:
%fs ls "dbfs:/Volumes"


When working with Databricks Unity Catalog, one powerful capability is linking it to external storage locations such as Azure Data Lake Storage (ADLS Gen2).

Why do this?
	•	Centralized governance over data stored outside the managed catalog
	•	Fine-grained access control (tables, schemas, files)
	•	Better auditing and lineage tracking with Purview integration

🔑 Steps Overview
1️⃣ Create an Azure Storage Container in ADLS Gen2.
2️⃣ Grant access to the Databricks service principal via Storage Blob Data Contributor role at the container or folder level.
3️⃣ In Databricks, use:

CREATE EXTERNAL LOCATION my_ext_loc
URL 'abfss://container@storageaccount.dfs.core.windows.net/folder'
WITH STORAGE CREDENTIAL my_storage_cred;

4️⃣ Map schemas and tables to this location or directly query the files using Unity Catalog’s security model.

✨ Benefits
	•	Keep raw/curated data in ADLS while still applying Unity Catalog governance
	•	Reduce duplication — govern where the data lives
	•	Enable secure cross-workspace data access

In [0]:
%sql 
select * from employees

In [0]:
%sql

UPDATE employees 
SET salary = salary + 100
WHERE name LIKE "A%"

In [0]:
%sql
RESTORE TABLE employees TO VERSION AS OF 4

In [0]:
%sql
OPTIMIZE employees
ZORDER BY id

In [0]:
%sql
DESCRIBE DETAIL employees

In [0]:
%sql 
VACUUM employees

In [0]:
%sql 
VACUUM employees RETAIN 0 HOURS

In [0]:
%sql 
drop table employees

bring data from external storage s3 bucket 

In [0]:
%sql
CREATE TABLE external_default
  (width INT, length INT, height INT)
LOCATION 'dbfs:/mnt/demo/external_default';

In [0]:
%sql 
DESCRIBE EXTENDED employees

In [0]:
%sql
INSERT INTO external_default
VALUES (3 INT, 2 INT, 1 INT)


In [0]:
%sql 
CREATE SCHEMA new_deafault

Views Comcepts

- temporary views (Notebook session ) 
- Global Views 

In [0]:
%sql 
CREATE TABLE IF NOT EXISTS smartphones
(id INT, name STRING, brand STRING, year INT);

INSERT INTO smartphones
VALUES (1, 'iPhone 14', 'Apple', 2022),
      (2, 'iPhone 13', 'Apple', 2021),
      (3, 'iPhone 6', 'Apple', 2014),
      (4, 'iPad Air', 'Apple', 2013),
      (5, 'Galaxy S22', 'Samsung', 2022),
      (6, 'Galaxy Z Fold', 'Samsung', 2022),
      (7, 'Galaxy S9', 'Samsung', 2016),
      (8, '12 Pro', 'Xiaomi', 2022),
      (9, 'Redmi 11T Pro', 'Xiaomi', 2022),
      (10, 'Redmi Note 11', 'Xiaomi', 2021)


In [0]:
%sql
show tables

In [0]:
%sql 
CREATE VIEW view_apple_phones
AS  SELECT * 
    FROM smartphones 
    WHERE brand = 'Apple';

In [0]:
%sql
SELECT * FROM view_apple_phones;

In [0]:
%sql 
CREATE TEMP VIEW temp_view_phones_brands
AS  SELECT DISTINCT brand
    FROM smartphones;

In [0]:
%sql
SELECT * FROM temp_view_phones_brands;

In [0]:
%sql 

CREATE GLOBAL TEMP VIEW global_temp_view_latest_phones
AS SELECT * FROM smartphones
    WHERE year > 2020
    ORDER BY year DESC;